# Test model submission workflow

This is a notebook to run the GitHub *backend* of M@TE.
It is useful for debugging the *backend* when M@TE has errors fetching metadata.

The *backend* parses the initial input from the GitHub issue by the M@TE submitter and attempts to enrich the metadata further by using API calls to 3rd party services.
The results are then converted into a JSON file and the original README.

This is orchestrated by the functions that this script calls.

To run this script on Issues submitted in M@TE one will need to use the GitHub python API to give themselves authenication to read the GitHub Issues. Do this by following. 

https://github-app-tutorial.readthedocs.io/en/latest/gh-api-cmd-line.html

*Note*: GitHub tokens are only valid for certain amount of time, 1 month. So you will often have to recreate this step. \
The following *assumes* your GH_CREDS is available via the environment variable GH_CREDS 

In [1]:
import os
import sys
import re
import requests
import subprocess
import pandas as pd
from IPython.display import Markdown
import json
import urllib
import string
import random
import copy
from github import Github, Auth
from secret import GH_CREDS
from pprint import pprint


In [2]:
#add path to workflow
sys.path.append("../scripts/") 
#test import
from crosswalk_mappings import *

## Access the original submission (issue) based on number

In [3]:
auth = Auth.Token(GH_CREDS)
g = Github(auth=auth)
g.get_user().login
repo = g.get_repo('ModelAtlasofTheEarth/Model_Submission')

In [4]:
# get issue based on issue number (https://github.com/ModelAtlasofTheEarth/model_submission/issues)
issue_number = 53
issue = repo.get_issue(number = issue_number)

## Manually extract the issue.body into a dictionary

This represents the "first level" parsing, 

* converts the issue Markdown body into a Python dictionary.
* Each section heading in the GitHub Model Submission template becomes a dictionary key
* the user’s response under each heading becomes the corresponding dictionary value.


In [5]:
## note that in the Github workflow - this bit happens within 'parse_issue'

regex = r"### *(?P<key>.*?)\s*[\r\n]+(?P<value>[\s\S]*?)(?=###|$)"
data = dict(re.findall(regex, issue.body))

In [7]:
data['submitter ORCID (or name)']

'https://orcid.org/0000-0003-4515-9296\n\n'

In [ ]:
#issue.body

## Parse issue 

ParseIssue Function – Workflow Summary

- `ParseIssue` is the second processing layer and performs the main “heavy lifting.”
- It acts as a wrapper/orchestrator for many subordinate functions that structure and normalise the issue data.
- Input: the dictionary produced from the first step (where GitHub template headings become keys).
- Output: a set of structured sub-dictionaries that will later be mapped into a formal JSON metadata document (e.g., an RO-Crate).

- `ParseIssue` calls a range of helper functions to:
  - clean and normalise fields
  - split complex responses into nested structures
  - validate identifiers
  - prepare fields for downstream conversion into RO-Crate entities

- A key part of this step involves identifier resolution.
  - Functions like `GetRecord` are used to retrieve metadata from external sources.
  - `GetRecord` is generic but must know the record type (e.g., DOI, ORCID, ROR, funder IDs).
  - Depending on the type, it dispatches to record-specific parsers such as:
    - `ParseOrganization`
    - `ProcessFundingData`
    - `ParsePerson`
    - etc.

- These functions may make external API calls (e.g., DOI APIs, ORCID, Crossref, ROR).
- API responses then need to be parsed, normalised, and converted into internal structures compatible with the final metadata schema.

Overall: `ParseIssue` is the orchestration layer that turns a loosely structured human-written GitHub issue into a fully structured internal metadata model, ready for conversion into RO-Crate JSON. 


In [8]:
from parse_issue import parse_issue 


In [9]:
issue_dict, log = parse_issue(issue)

https://api.crossref.org/works/https://doi.org/10.5194/esurf-12-301-2024
https://doi.org/10.5281/zenodo.1069573
https://pub.orcid.org/v3.0/0000-0003-4515-9296


Check what's in "issue_dict"

In [10]:
issue_dict['publication']

{'@type': 'ScholarlyArticle',
 '@id': 'https://doi.org/10.5194/esurf-12-301-2024',
 'name': 'Flexural isostatic response of continental-scale deltas to climatically driven sea level changes',
 'isPartOf': ({'@type': 'PublicationIssue',
   'issueNumber': '1',
   'datePublished': '2024-2-1',
   'isPartOf': {'@type': ['PublicationVolume', 'Periodical'],
    'name': ['Earth Surface Dynamics'],
    'issn': ['2196-632X'],
    'volumeNumber': '12',
    'publisher': 'Copernicus GmbH'}},),
 'author': [{'@type': 'Person',
   '@id': 'https://orcid.org/0000-0002-1270-4377',
   'givenName': 'Sara',
   'familyName': 'Polanco'},
  {'@type': 'Person', 'givenName': 'Mike', 'familyName': 'Blum'},
  {'@type': 'Person',
   '@id': 'https://orcid.org/0000-0001-6095-7689',
   'givenName': 'Tristan',
   'familyName': 'Salles'},
  {'@type': 'Person', 'givenName': 'Bruce C.', 'familyName': 'Frederick'},
  {'@type': 'Person',
   '@id': 'https://orcid.org/0000-0002-2594-6965',
   'givenName': 'Rebecca',
   'famil

In [11]:
issue_dict["software"]

{'@context': 'http://schema.org',
 '@type': 'SoftwareSourceCode',
 '@id': 'https://doi.org/10.5281/zenodo.1069573',
 'identifier': {'@type': 'PropertyValue',
  'propertyID': 'URL',
  'value': 'https://zenodo.org/record/1069573'},
 'url': 'https://zenodo.org/record/1069573',
 'name': 'badlands-model/pyBadlands: stable release version 2.0.0',
 'author': [{'name': 'Tristan Salles',
   'givenName': 'Tristan',
   'familyName': 'Salles',
   'affiliation': {'@type': 'Organization', 'name': 'University of Sydney'},
   '@type': 'Person'},
  {'name': 'Ian Howson',
   'givenName': 'Ian',
   'familyName': 'Howson',
   '@type': 'Person'}],
 'description': 'This release adds new capabilities to the code: multiple formulations for fluvial processes from detachment to transport limited cases are implemented, long-term evolution of longshore currents and induced sediment transport is simulated based on linear wave theory fuzzy logic approach is used to estimate carbonate growth based on depth, wave ene

In [12]:
len(issue_dict['creators'])

9

## Testing the subordinate functions

In [13]:
from request_utils import get_record
from parse_metadata_utils import  parse_organization
from parse_utils import process_funding_data, get_funders, parse_organization

In [14]:
issue_dict.keys()

dict_keys(['publication', 'software', 'submitter', 'creators', 'proposed_slug', 'slug', 'for_codes', 'license', 'model_category', 'model_status', 'title', 'abstract', 'description', 'scientific_keywords', 'funder', 'funding', 'embargo', 'include_model_code', 'model_code_inputs', 'include_model_output', 'model_output_data', 'computer_resource', 'landing_image', 'animation', 'graphic_abstract', 'model_setup_figure', 'model_setup_description'])

In [21]:
data['publication DOI'].strip()

'https://doi.org/10.5194/esurf-12-301-2024'

In [27]:
data['contributing authors'].strip()

'```text\n\n```'

In [29]:
data.keys()

dict_keys(['submitter ORCID (or name)', 'slug', 'license', 'alternative license URL', 'description', 'model category', 'model status', 'publication DOI', 'title', 'contributing authors', 'abstract', 'scientific keywords', 'funder', 'model links', 'include model inputs?', 'model input usage, a.k.a reproducibility', 'include model output?', 'model output notes', 'model output size', 'model embargo?', 'software DOI/URI', 'Software source repository', 'Software version used.', 'Provide the name of the software (e.g. Underworld, ASPECT, Badlands, OpenFOAM)', 'software authors', 'software & algorithm keywords', 'computer URI/DOI', 'add landing page image', 'add an animation', 'add a graphic abstract figure', 'add a model setup figure', 'add a description of your model setup', 'Provide any feedback on the model submission process?'])

In [23]:
get_record('publication', data['publication DOI'].strip())

https://api.crossref.org/works/https://doi.org/10.5194/esurf-12-301-2024


({'status': 'ok',
  'message-type': 'work',
  'message-version': '1.0.0',
  'message': {'indexed': {'date-parts': [[2026, 5, 13]],
    'date-time': '2026-05-13T03:36:20Z',
    'timestamp': 1778643380341,
    'version': '3.51.4'},
   'reference-count': 72,
   'publisher': 'Copernicus GmbH',
   'issue': '1',
   'license': [{'start': {'date-parts': [[2024, 2, 1]],
      'date-time': '2024-02-01T00:00:00Z',
      'timestamp': 1706745600000},
     'content-version': 'vor',
     'delay-in-days': 0,
     'URL': 'https://creativecommons.org/licenses/by/4.0/'}],
   'funder': [{'DOI': '10.13039/501100000923',
     'name': 'Australian Research Council',
     'doi-asserted-by': 'publisher',
     'award': ['IH130200012'],
     'award-info': [{'award-number': ['IH130200012']}],
     'id': [{'id': '10.13039/501100000923',
       'id-type': 'DOI',
       'asserted-by': 'publisher'}]},
    {'DOI': '10.13039/501100001782',
     'name': 'University of Melbourne',
     'doi-asserted-by': 'publisher',
    

In [30]:
print(data["funder"])
funders_string = data["funder"]
funders_dict = process_funding_data(funders_string)
print(funders_dict)

```text

```


{'funders': [{'@type': 'Organization', 'name': '```text'}, {'@type': 'Organization', 'name': '```'}], 'funding': []}


# Test get_record/parse_organization  directly 

* Here we are testing the get_record by handing in a ROR id
* then using parse_organization to handle that record

In [31]:
# Test the underlying API directly
print("\n🔹 Testing get_record() directly on one ROR ID...")
record, log = get_record("organization", "05mmh0f86")
print("Record keys:", list(record.keys()))
print("Log:", log)

# Test parsing that record into schema.org format
print("\n🔹 Testing parse_organization()...")
parsed, parse_log = parse_organization(record)
pprint(parsed)
print("Parse log:", parse_log)


🔹 Testing get_record() directly on one ROR ID...
https://api.ror.org/organizations/05mmh0f86
Record keys: ['admin', 'domains', 'established', 'external_ids', 'id', 'links', 'locations', 'names', 'relationships', 'status', 'types']
Log: Could not fetch metadata with application/ld+json from https://api.ror.org/organizations/05mmh0f86: 406 Client Error: Not Acceptable for url: https://api.ror.org/organizations/05mmh0f86



🔹 Testing parse_organization()...
{'@id': 'https://ror.org/05mmh0f86',
 '@type': 'Organization',
 'additionalType': ['funder', 'government'],
 'name': 'ARC',
 'url': 'https://www.arc.gov.au'}
Parse log: 


# Converting the issue_dict

In [32]:
from crosswalks import dict_to_report, dict_to_metadata
from ro_crate_utils import load_entity_template, recursively_filter_key
from crosswalks import dict_to_report, dict_to_metadata, dict_to_ro_crate_mapping, default_issue_entity_mapping_list
from ro_crate_utils import load_crate_template, defaults_and_customise_ro_crate


## Issue to report

In [33]:

the_md_report = dict_to_report(issue_dict)

## Issue to metadata

In [34]:
# Perform a deep copy of the issue_dict to avoid modifying the input dictionary
issue_dict_copy = copy.deepcopy(issue_dict)

#this takes the issue_dict and simplifies entities (e.g. @Type=Person) using templates defined at:
#https://github.com/ModelAtlasofTheEarth/metadata_schema/blob/main/mate_ro_crate/type_templates.json

entity_template = load_entity_template()
recursively_filter_key(issue_dict_copy, entity_template)

#this maps the issue_dict_copy to the ro_crate
ro_crate = dict_to_metadata(issue_dict_copy, flat_compact_crate=False)

JSON-LD data loaded successfully.
JSON-LD data loaded successfully.
JSON-LD data loaded successfully.


In [36]:
the_md_report

'## Section 1: Summary of your model   \n\n**Model Submitter:**  \n\nJulian Giordani ([0000-0003-4515-9296](https://orcid.org/0000-0003-4515-9296))\n\n**Model Creator(s):**  \n\n- Sara Polanco ([0000-0002-1270-4377](https://orcid.org/0000-0002-1270-4377))  \n- Mike Blum   \n- Tristan Salles ([0000-0001-6095-7689](https://orcid.org/0000-0001-6095-7689))  \n- Bruce C. Frederick   \n- Rebecca Farrington ([0000-0002-2594-6965](https://orcid.org/0000-0002-2594-6965))  \n- Xuesong Ding ([0000-0003-3693-932X](https://orcid.org/0000-0003-3693-932X))  \n- Ben Mather ([0000-0003-3566-1557](https://orcid.org/0000-0003-3566-1557))  \n- Claire Mallard ([0000-0003-2595-2414](https://orcid.org/0000-0003-2595-2414))  \n- Louis Moresi ([0000-0003-3685-174X](https://orcid.org/0000-0003-3685-174X))  \n  \n**Model slug:**  \n\n`` \n\n(this will be the name of the model repository when created) \n\n**Model name:**  \n\n_Flexural isostatic response of continental-scale deltas to climatically driven sea leve

In [35]:
issue_dict

{'publication': {'@type': 'ScholarlyArticle',
  '@id': 'https://doi.org/10.5194/esurf-12-301-2024',
  'name': 'Flexural isostatic response of continental-scale deltas to climatically driven sea level changes',
  'isPartOf': ({'@type': 'PublicationIssue',
    'issueNumber': '1',
    'datePublished': '2024-2-1',
    'isPartOf': {'@type': ['PublicationVolume', 'Periodical'],
     'name': ['Earth Surface Dynamics'],
     'issn': ['2196-632X'],
     'volumeNumber': '12',
     'publisher': 'Copernicus GmbH'}},),
  'author': [{'@type': 'Person',
    '@id': 'https://orcid.org/0000-0002-1270-4377',
    'givenName': 'Sara',
    'familyName': 'Polanco'},
   {'@type': 'Person', 'givenName': 'Mike', 'familyName': 'Blum'},
   {'@type': 'Person',
    '@id': 'https://orcid.org/0000-0001-6095-7689',
    'givenName': 'Tristan',
    'familyName': 'Salles'},
   {'@type': 'Person', 'givenName': 'Bruce C.', 'familyName': 'Frederick'},
   {'@type': 'Person',
    '@id': 'https://orcid.org/0000-0002-2594-6965'

In [ ]:
#dict_to_metadata?

In [38]:
ro_crate

'{"@context": ["https://www.researchobject.org/ro-crate/1.1/context.jsonld", "https://raw.githubusercontent.com/codemeta/codemeta/master/codemeta.jsonld"], "@graph": [{"@type": "CreativeWork", "@id": "ro-crate-metadata.json", "conformsTo": {"@id": "https://w3id.org/ro/crate/1.1"}, "about": {"@id": "./"}, "description": "RO-Crate Metadata File Descriptor (this file)"}, {"@id": "./", "identifier": [], "@type": "Dataset", "name": "Flexural isostatic response of continental-scale deltas to climatically driven sea level changes", "alternateName": "", "datePublished": "", "description": "This is a dummy submission to test every fields and how it interacts with the new M@TE 2 system", "abstract": "The interplay between climate-forced sea level change, erosional and depositional processes, and flexural isostasy in deep time on passive margin deltas remains poorly understood. We performed a series of conceptual simulations to investigate flexural isostatic responses to high-frequency fluctuatio

## Further notes

`dict_to_metadata` loads a loads the base RO-Crate skeleton, containing:

* @context
* top-level @graph with placeholders
* default MATE-specific RO-Crate metadata

- the mapping_list (list) argument of dict_to_metadata: A list of mappings that define how elements in the issue dictionary correspond to elements in the RO-Crate structure.

In [ ]:
#dict_to_metadata produces a string, suitable for writign the rocrate to file. 
# we can also load this a dictionary

rocratedict = json.loads(ro_crate)

rocratedict['@graph'][1]['citation']